# 03 — Models

> **Mirrors `src/worldcup/models/` (`baseline.py`, `xgboost_model.py`, `poisson.py`, `calibration.py`, `dixon_coles.py`).**
> Readable walkthrough that reproduces the same numbers; the code is the source of truth.

In [1]:
import pandas as pd
from worldcup.data.load import load_results
from worldcup.features.build import build_features
from worldcup.models.baseline import evaluate_baseline
from worldcup.models.xgboost_model import evaluate_xgb, train_production_xgb, feature_importances
from worldcup.models.poisson import evaluate_poisson, train_poisson, predict_match
from worldcup.models.calibration import evaluate_calibration

feats = build_features(load_results())

## Match-outcome models (same time-based test split)

In [2]:
b = evaluate_baseline(feats)
x = evaluate_xgb(feats)
p = evaluate_poisson(feats)
pd.DataFrame({
    "model": ["Logistic", "XGBoost", "Poisson"],
    "accuracy": [b["accuracy"], x["accuracy"], p["accuracy"]],
    "log_loss": [b["log_loss"], x["log_loss"], p["log_loss"]],
}).round(3)

,model,accuracy,log_loss
0,Logistic,0.599,0.875
1,XGBoost,0.598,0.877
2,Poisson,0.597,0.880


The three tie — the signal is largely linear (`elo_diff` dominates importance):

In [3]:
feature_importances(train_production_xgb(feats, save=False))

{'elo_diff': 0.7436106204986572,
 'form_points_diff': 0.0415116623044014,
 'form_gd_diff': 0.1508840024471283,
 'neutral': 0.0639936774969101}

## Calibration (are the probabilities trustworthy?)

In [4]:
cal = evaluate_calibration(feats)
print(f"Brier score: {cal['brier']:.3f}  (uniform baseline 0.667)")

Brier score: 0.514  (uniform baseline 0.667)


## Poisson scorelines + Dixon-Coles

In [5]:
pois = train_poisson(feats, save=False)
demo = predict_match(pois, {"elo_diff": 200.0, "neutral": 1})
s = demo["most_likely_score"]
print(f"Favourite (elo_diff=200): P(H/D/A)={ {k: round(v,3) for k,v in demo['outcome'].items()} }")
print(f"Most likely score: {s['home']}-{s['away']} ({s['prob']:.1%})")

Favourite (elo_diff=200): P(H/D/A)={'H': 0.64, 'D': 0.202, 'A': 0.158}
Most likely score: 2-0 (10.8%)


In [6]:
from worldcup.models.dixon_coles import DixonColesModel
dc = DixonColesModel().fit(load_results())
print(f"Dixon-Coles: {len(dc.teams)} teams, home_adv={dc.home_adv:.3f}, rho={dc.rho:.3f}")
dc.predict_match("Argentina", "Brazil")["outcome"]

Dixon-Coles: 217 teams, home_adv=0.265, rho=-0.035


{'H': 0.4505233216272063, 'D': 0.2873862113107296, 'A': 0.2620904670620642}